# Maestro de TVL por protocolo (append-only)
Toma las carpetas de ejecuciones **en orden cronológico** y arma un maestro que **solo agrega fechas nuevas**. Nunca pisa lo ya cargado.

Salidas (en `BASE/maestro/`): `maestro_tvl_largo.csv`, `revisiones.csv`, `estado_protocolos.csv`, `tb_agregado.csv`, `manifest.json`.

Clave del maestro: `(id del protocolo, chain)`. El `id` (no el slug) evita duplicar protocolos renombrados (ej. kamui → kamui-finance).

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import os, json, glob, hashlib, datetime as dt
import pandas as pd, numpy as np

# ── CONFIGURACIÓN ─────────────────────────────────────────
BASE = os.environ.get("TESIS_OUT_DIR", "/content/drive/MyDrive/Tesis/outputs_y_data")
EJECUCIONES = ["ejecucion_3", "ejecucion_4"]      # orden cronológico
TAG_TB = "Treasury Bills"                         # tag del agregado de prueba
COL_TB = "rwa_tipo_treasurybills"
UMBRAL_REV = 0.01                                 # diferencia relativa que cuenta como revisión
DIR_OUT = os.path.join(BASE, "maestro")
os.makedirs(DIR_OUT, exist_ok=True)

In [7]:
# ── LECTURA DE CRUDOS (misma regla que el Job 03) ─────────
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def a_fecha(ts):
    return pd.Timestamp(pd.to_datetime(int(ts), unit="s").date())

def series_por_chain(data):
    """chainTvls[chain]['tvl'] sin claves con '-' (borrowed, staking...). Última obs. del día."""
    out = {}
    for ch, c in (data.get("chainTvls") or {}).items():
        if "-" in str(ch) or not isinstance(c, dict):
            continue
        obs = c.get("tvl")
        if not isinstance(obs, list):
            continue
        p = {}
        for o in obs:
            if not isinstance(o, dict) or o.get("date") is None or o.get("totalLiquidityUSD") is None:
                continue
            try:
                p[a_fecha(o["date"])] = float(o["totalLiquidityUSD"])
            except (TypeError, ValueError):
                continue
        if p:
            out[str(ch)] = pd.Series(p).sort_index()
    return out

def leer_ejecucion(nombre):
    raw = os.path.join(BASE, nombre, "data", "raw")
    fechas = sorted(d for d in os.listdir(raw) if os.path.isdir(os.path.join(raw, d)))
    if len(fechas) != 1:
        raise RuntimeError(f"[DEBUG][ERROR] {nombre}: se esperaba 1 carpeta de fecha en raw, hay {fechas}")
    fecha = fechas[0]; d = os.path.join(raw, fecha)
    protos, hashes = {}, {}
    for ruta in sorted(glob.glob(os.path.join(d, "protocol_*.json"))):
        with open(ruta, encoding="utf-8") as f:
            data = json.load(f)
        pid = str(data.get("id"))
        protos[pid] = dict(id=pid, slug=os.path.basename(ruta)[9:-5], name=data.get("name"),
                           tags=data.get("tags") or [], category=data.get("category"),
                           series=series_por_chain(data))
        hashes[os.path.basename(ruta)] = sha256(ruta)
    print(f"[{nombre}] extracción {fecha}: {len(protos)} protocolos, "
          f"{sum(1 for p in protos.values() if p['series'])} con serie")
    return fecha, protos, hashes

EJ = {n: leer_ejecucion(n) for n in EJECUCIONES}

[ejecucion_3] extracción 2026-10-27: 187 protocolos, 129 con serie
[ejecucion_4] extracción 2026-11-21: 190 protocolos, 116 con serie


In [8]:
# ── CONSTRUCCIÓN DEL MAESTRO (append-only) ────────────────
maestro = {}      # (id, chain) -> DataFrame[tvl, vintage]
meta = {}         # id -> dict(slug, name, tags, ...)
revisiones = []   # comparación en fechas comunes
primera = True
for nombre in EJECUCIONES:
    fecha, protos, _ = EJ[nombre]
    for pid, p in protos.items():
        meta.setdefault(pid, dict(slugs=[], name=p["name"], tags=p["tags"], category=p["category"],
                                  primera_ejecucion=nombre))
        if p["slug"] not in meta[pid]["slugs"]:
            meta[pid]["slugs"].append(p["slug"])
        meta[pid]["tags"] = p["tags"] or meta[pid]["tags"]
        for ch, s in p["series"].items():
            k = (pid, ch)
            if k not in maestro:
                maestro[k] = pd.DataFrame({"tvl": s, "vintage": fecha})
                continue
            m = maestro[k]
            ult = m.index.max()
            # 1) control de cambios en fechas comunes (excluye el último día del maestro:
            #    es un valor "en vivo" del día de la extracción anterior)
            com = m.index.intersection(s.index)
            com = com[com < ult]
            if len(com):
                rel = (s.reindex(com) - m["tvl"].reindex(com)).abs() / m["tvl"].reindex(com).abs().clip(lower=1)
                revisiones.append(dict(ejecucion=nombre, id=pid, chain=ch, n_comunes=len(com),
                                       n_revisadas=int((rel > UMBRAL_REV).sum()),
                                       max_rel=float(rel.max())))
            # 2) append: solo fechas posteriores al último dato del maestro
            nuevo = s[s.index > ult]
            if len(nuevo):
                maestro[k] = pd.concat([m, pd.DataFrame({"tvl": nuevo, "vintage": fecha})])
    # chains que estaban en el maestro y esta ejecución ya no devuelve
    for (pid, ch) in list(maestro):
        pr = protos.get(pid)
        if pr is None or ch not in pr["series"]:
            revisiones.append(dict(ejecucion=nombre, id=pid, chain=ch, n_comunes=0, n_revisadas=0,
                                   max_rel=np.nan, nota="chain sin datos en esta ejecución (se conserva historia)"))
    primera = False

largo = pd.concat([m.assign(id=k[0], chain=k[1]).rename_axis("fecha").reset_index() for k, m in maestro.items()],
                  ignore_index=True)
largo["slug"] = largo["id"].map(lambda i: meta[i]["slugs"][-1])
largo = largo[["id", "slug", "chain", "fecha", "tvl", "vintage"]].sort_values(["id", "chain", "fecha"])
df_rev = pd.DataFrame(revisiones)
print(f"Maestro: {len(maestro)} series (protocolo, chain), {len(largo):,} filas")
print(df_rev.groupby("ejecucion")[["n_comunes", "n_revisadas"]].sum() if len(df_rev) else "sin revisiones")

Maestro: 275 series (protocolo, chain), 186,807 filas
             n_comunes  n_revisadas
ejecucion                          
ejecucion_4     163340            0


In [9]:
# ── ESTADO POR PROTOCOLO ──────────────────────────────────
ult_nombre = EJECUCIONES[-1]
_, protos_ult, _ = EJ[ult_nombre]
filas = []
for pid, mt in meta.items():
    en_ult = protos_ult.get(pid)
    activo = bool(en_ult and en_ult["series"])
    fechas = largo.loc[largo["id"] == pid, "fecha"]
    estado = ("activo" if activo else
              "sin_serie_en_ultima_ejecucion" if fechas.size else "nunca_tuvo_serie")
    filas.append(dict(id=pid, slugs="|".join(mt["slugs"]), name=mt["name"], tags=",".join(mt["tags"]),
                      estado=estado, primera_ejecucion=mt["primera_ejecucion"],
                      primera_fecha=fechas.min() if fechas.size else None,
                      ultima_fecha=fechas.max() if fechas.size else None,
                      renombrado=len(mt["slugs"]) > 1))
df_estado = pd.DataFrame(filas).sort_values("estado")
print(df_estado["estado"].value_counts().to_string())
print("Renombrados:", df_estado.loc[df_estado["renombrado"], "slugs"].tolist())

estado
activo                           116
nunca_tuvo_serie                  61
sin_serie_en_ultima_ejecucion     14
Renombrados: ['kamui|kamui-finance']


In [10]:
# ── AGREGADO DE PRUEBA: Treasury Bills (método del Job 03) ─
def densificar(s):
    return s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="D")).ffill()

def agregar(series):
    v = [s for s in series if len(s)]
    g = pd.date_range(min(s.index.min() for s in v), max(s.index.max() for s in v), freq="D")
    t = pd.Series(0.0, index=g)
    for s in v:
        t = t.add(densificar(s).reindex(g).fillna(0.0), fill_value=0.0)
    return t

ids_tb = {i for i, m in meta.items() if TAG_TB in m["tags"]}
activos = set(df_estado.loc[df_estado["estado"] == "activo", "id"])
tb_hist = agregar([m["tvl"] for (i, c), m in maestro.items() if i in ids_tb])
tb_act = agregar([m["tvl"] for (i, c), m in maestro.items() if i in ids_tb and i in activos])
tb = pd.DataFrame({COL_TB + "_con_historial": tb_hist, COL_TB + "_solo_activos": tb_act}).rename_axis("fecha")
tb["salto_diario_historial"] = tb.iloc[:, 0].pct_change()
print(f"{len(ids_tb)} protocolos con tag '{TAG_TB}'; agregado {tb.index.min().date()} → {tb.index.max().date()}")
saltos = tb[tb["salto_diario_historial"] < -0.10]
print("Caídas diarias > 10 % (revisar si son salidas de fuente):")
print(saltos.tail(5).to_string() if len(saltos) else "ninguna")

39 protocolos con tag 'Treasury Bills'; agregado 2021-04-22 → 2026-11-21
Caídas diarias > 10 % (revisar si son salidas de fuente):
            rwa_tipo_treasurybills_con_historial  rwa_tipo_treasurybills_solo_activos  salto_diario_historial
fecha                                                                                                        
2026-07-16                          1.902031e+09                         1.793668e+09               -0.126325
2026-09-18                          1.436090e+09                         1.229003e+09               -0.150753
2026-10-28                          1.126167e+09                         1.126167e+09               -0.167160
2026-10-30                          9.013521e+08                         9.013521e+08               -0.164534
2026-11-13                          6.835597e+08                         6.835597e+08               -0.124623


In [11]:
# ── CONTROL: el maestro reproduce la 1.ª ejecución ────────
# Si octubre (1.ª ejecución) se reproduce exacto en sus fechas, el append no alteró lo anterior.
ref_ruta = os.path.join(BASE, EJECUCIONES[0], "data", "processed", "series_rwa_largas.csv")
if os.path.exists(ref_ruta):
    ref = pd.read_csv(ref_ruta, index_col=0, parse_dates=True)[COL_TB].dropna()
    dif = (tb.iloc[:, 0].reindex(ref.index) - ref).abs()
    print(f"Referencia {EJECUCIONES[0]}: {len(ref)} fechas, diferencia máxima = {dif.max():,.4f} USD")
    assert dif.max() < 1.0, "[DEBUG][ERROR] el maestro NO reproduce la primera ejecución"
    print("OK: el maestro reproduce exactamente la primera ejecución.")
else:
    print("[DEBUG][ALERTA] no se encontró la serie de referencia de la 1.ª ejecución; control omitido")

Referencia ejecucion_3: 2015 fechas, diferencia máxima = 0.0000 USD
OK: el maestro reproduce exactamente la primera ejecución.


In [12]:
# ── GUARDADO + MANIFEST ───────────────────────────────────
rutas = {
    "maestro_tvl_largo.csv": largo, "revisiones.csv": df_rev,
    "estado_protocolos.csv": df_estado, "tb_agregado.csv": tb.reset_index(),
}
manifest = {"generado": dt.datetime.now(dt.timezone.utc).isoformat(), "ejecuciones": {}, "salidas": {}}
for n in EJECUCIONES:
    manifest["ejecuciones"][n] = {"fecha_extraccion": EJ[n][0], "sha256_crudos": EJ[n][2]}
for nombre, df in rutas.items():
    r = os.path.join(DIR_OUT, nombre); df.to_csv(r, index=False)
    manifest["salidas"][nombre] = {"filas": int(len(df)), "sha256": sha256(r)}
with open(os.path.join(DIR_OUT, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=1, ensure_ascii=False)
print("Guardado en", DIR_OUT)
for k, v in manifest["salidas"].items():
    print(f"  {k:26} {v['filas']:>8,} filas  sha256 {v['sha256'][:12]}")

Guardado en /content/drive/MyDrive/Tesis/outputs_y_data/maestro
  maestro_tvl_largo.csv       186,807 filas  sha256 58f663f40e63
  revisiones.csv                  270 filas  sha256 b8bbbe686672
  estado_protocolos.csv           191 filas  sha256 b4ac59b75bb2
  tb_agregado.csv               2,040 filas  sha256 cfb7154baf64


In [13]:
m = pd.read_csv(DIR_OUT + "/maestro_tvl_largo.csv", parse_dates=["fecha"])
r = pd.read_csv(DIR_OUT + "/revisiones.csv")
e = pd.read_csv(DIR_OUT + "/estado_protocolos.csv")
print("duplicados (id,chain,fecha):", m.duplicated(["id","chain","fecha"]).sum())   # debe ser 0
print(m.groupby("vintage")["fecha"].agg(["count","min","max"]))
print(e["estado"].value_counts())
print("revisadas >1%:", r["n_revisadas"].sum(), "de", r["n_comunes"].sum())
print(r[r["n_revisadas"] > 0].sort_values("max_rel", ascending=False).head(10))
print(e[e["estado"] != "activo"][["slugs","tags","estado","ultima_fecha"]].to_string())

/tmp/ipykernel_1567/166681701.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  m = pd.read_csv(DIR_OUT + "/maestro_tvl_largo.csv", parse_dates=["fecha"])


duplicados (id,chain,fecha): 0
             count        min        max
vintage                                 
2026-10-27  180927 2021-04-22 2026-10-27
2026-11-21    5880 2026-10-23 2026-11-21
estado
activo                           116
nunca_tuvo_serie                  61
sin_serie_en_ultima_ejecucion     14
Name: count, dtype: int64
revisadas >1%: 0 de 163340
Empty DataFrame
Columns: [ejecucion, id, chain, n_comunes, n_revisadas, max_rel, nota]
Index: []
                                                         slugs                                                                 tags                         estado ultima_fecha
116                                                    xstocks                                                        Stocks & ETFs               nunca_tuvo_serie          NaN
117                                               xend-finance                                                         Crowdfunding               nunca_tuvo_serie          NaN
118      